# Hito 2 — Modelado: MLP, LSTM y BiLSTM

Se han seleccionado tres modelos diferentes con el objetivo de comparar distintos enfoques para la predicción de series temporales financieras.

- **MLP (Multi-Layer Perceptron)** se utiliza como modelo base o *baseline*. Aunque no está diseñado específicamente para series temporales, permite establecer un punto de referencia para evaluar la mejora aportada por arquitecturas recurrentes.

- **LSTM (Long Short-Term Memory)** se selecciona por su capacidad para capturar dependencias temporales de largo plazo presentes en los datos financieros. A diferencia de redes neuronales tradicionales, las LSTM incorporan mecanismos de memoria que les permiten conservar información relevante de observaciones anteriores.

- **BiLSTM (Bidirectional LSTM)** amplía el funcionamiento de las LSTM procesando las secuencias tanto hacia adelante como hacia atrás durante el entrenamiento. Esto permite capturar relaciones temporales más complejas y obtener una representación más completa de los patrones presentes en los datos.

### Configuración de los modelos

- MLP con capas ocultas `(100, 50)` como modelo baseline.
- LSTM de 64 unidades con `Dropout(0.2)` y optimizador `Adamax`.
- BiLSTM de 64 unidades con `Dropout(0.2)` y optimizador `Adam`.
- Early Stopping (`patience=10`) monitorizando `val_loss` para evitar sobreajuste.
- `SEED = 42` para garantizar la reproducibilidad de los resultados.

Scripts que se han ejecutado:
- Carga los arrays de `datasets/train_test/`

In [2]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neural_network import MLPRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional


In [3]:
# Reproducibilidad
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
scaler = joblib.load("../datasets/train_test/scaler.pkl")
data_scaled = joblib.load("../datasets/train_test/data_scaled.pkl")

X_train = np.load("../datasets/train_test/X_train.npy")
X_test = np.load("../datasets/train_test/X_test.npy")

y_train = np.load("../datasets/train_test/y_train.npy")
y_test = np.load("../datasets/train_test/y_test.npy")

dates_train = np.load("../datasets/train_test/dates_train.npy")
dates_test = np.load("../datasets/train_test/dates_test.npy")


# Modelos

### Modelo 1: MLP

In [5]:
# MLP necesita datos 2D
X_train_mlp = X_train.reshape(X_train.shape[0], X_train.shape[1])
X_test_mlp = X_test.reshape(X_test.shape[0], X_test.shape[1])

model_mlp = MLPRegressor(
    hidden_layer_sizes=(100, 50),
    activation="relu",
    max_iter=500,
    random_state=42
)

model_mlp.fit(X_train_mlp, y_train.ravel())

pred_mlp = model_mlp.predict(X_test_mlp)

y_test_inv = scaler.inverse_transform(y_test)

pred_mlp_inv = scaler.inverse_transform(
    pred_mlp.reshape(-1, 1)
)

mae = mean_absolute_error(y_test_inv, pred_mlp_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, pred_mlp_inv))
r2 = r2_score(y_test_inv, pred_mlp_inv)

print(f"MAE: {mae:.2f} USD")
print(f"RMSE: {rmse:.2f} USD")
print(f"R2: {r2:.4f}")

MAE: 201.54 USD
RMSE: 291.64 USD
R2: 0.9081


### Modelo 2: LSTM

In [6]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# MODELO LSTM BÁSICO
# =========================

time_steps = X_train.shape[1]

model_lstm = Sequential([
    LSTM(
        units=64,
        input_shape=(time_steps, 1)
    ),
    Dropout(0.2),
    Dense(1)
])

model_lstm.compile(
    optimizer="adamax",
    loss="mse"
)

# =========================
# ENTRENAMIENTO
# =========================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model_lstm.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)

# =========================
# PREDICCIÓN
# =========================

pred_lstm = model_lstm.predict(X_test)

# =========================
# INVERSIÓN DEL ESCALADO
# =========================

y_test_inv = scaler.inverse_transform(np.array(y_test).reshape(-1, 1))
pred_lstm_inv = scaler.inverse_transform(np.array(pred_lstm).reshape(-1, 1))

# =========================
# MÉTRICAS
# =========================

mae = mean_absolute_error(y_test_inv, pred_lstm_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, pred_lstm_inv))
r2 = r2_score(y_test_inv, pred_lstm_inv)

print("\nLSTM")
print(f"MAE: {mae:.2f} USD")
print(f"RMSE: {rmse:.2f} USD")
print(f"R2: {r2:.4f}")

c:\Users\pedre\OneDrive\Documentos\ProyectoFinal\venv_proyectoFinal\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 11s 86ms/step - loss: 0.0034 - val_loss: 0.0052
Epoch 2/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - loss: 0.0014 - val_loss: 0.0020
Epoch 3/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - loss: 6.3890e-04 - val_loss: 3.6339e-04
Epoch 4/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - loss: 2.9343e-04 - val_loss: 8.5205e-05
Epoch 5/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 5s 64ms/step - loss: 2.2490e-04 - val_loss: 7.2584e-05
Epoch 6/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - loss: 1.9352e-04 - val_loss: 7.1907e-05
Epoch 7/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - loss: 1.7055e-04 - val_loss: 7.4642e-05
Epoch 8/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - loss: 1.7651e-04 - val_loss: 7.7470e-05
Epoch 9/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 3s 61ms/step - loss: 1.6756e-04 - val_loss: 7.0925e-05
Epoch 10/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 89ms/step - loss: 1.7041e-04 - val_loss: 8.0617e-05
Epoch 11/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - loss: 1.7097e-04 - 

### Modelo 3: BiLSTM

In [7]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# MODELO BiLSTM BÁSICO
# =========================

time_steps = X_train.shape[1]

model_bilstm = Sequential([
    Bidirectional(
        LSTM(64),
        input_shape=(time_steps, 1)
    ),
    Dropout(0.2),
    Dense(1)
])

model_bilstm.compile(
    optimizer="adam",
    loss="mse"
)

# =========================
# ENTRENAMIENTO
# =========================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=False
)

history = model_bilstm.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    shuffle=False,
    callbacks=[],
    verbose=1
)

# =========================
# PREDICCIÓN
# =========================

pred_bilstm = model_bilstm.predict(X_test)

# =========================
# INVERSIÓN ESCALADO
# =========================

y_test_inv = scaler.inverse_transform(
    np.array(y_test).reshape(-1, 1)
)

pred_bilstm_inv = scaler.inverse_transform(
    np.array(pred_bilstm).reshape(-1, 1)
)

# =========================
# MÉTRICAS
# =========================

mae = mean_absolute_error(y_test_inv, pred_bilstm_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, pred_bilstm_inv))
r2 = r2_score(y_test_inv, pred_bilstm_inv)

print("\nBiLSTM")
print(f"MAE: {mae:.2f} USD")
print(f"RMSE: {rmse:.2f} USD")
print(f"R2: {r2:.4f}")

c:\Users\pedre\OneDrive\Documentos\ProyectoFinal\venv_proyectoFinal\Lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 18s 121ms/step - loss: 0.0017 - val_loss: 8.5951e-04
Epoch 2/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 93ms/step - loss: 0.0020 - val_loss: 2.4178e-04
Epoch 3/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 90ms/step - loss: 0.0011 - val_loss: 2.2704e-04
Epoch 4/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 86ms/step - loss: 7.5556e-04 - val_loss: 1.5196e-04
Epoch 5/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 86ms/step - loss: 6.0896e-04 - val_loss: 1.2873e-04
Epoch 6/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 92ms/step - loss: 5.2935e-04 - val_loss: 1.2259e-04
Epoch 7/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 98ms/step - loss: 4.6006e-04 - val_loss: 1.2001e-04
Epoch 8/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 93ms/step - loss: 3.9356e-04 - val_loss: 1.1624e-04
Epoch 9/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 91ms/step - loss: 3.3741e-04 - val_loss: 1.2598e-04
Epoch 10/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 6s 99ms/step - loss: 3.0510e-04 - val_loss: 1.2326e-04
Epoch 11/100
46/46 ━━━━━━━━━━━━━━━━━━━━ 4s 90ms/step - loss: 2.6688e-

# Evaluación

In [8]:
def evaluar(nombre, y_real, y_pred):

    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    print(f"\n{nombre}")
    print(f"MAE: {mae:.2f} USD")
    print(f"RMSE: {rmse:.2f} USD")
    print(f"R2: {r2:.4f}")

evaluar("MLP", y_test_inv, pred_mlp_inv)
evaluar("LSTM", y_test_inv, pred_lstm_inv)
evaluar("BILSTM", y_test_inv, pred_bilstm_inv)


MLP
MAE: 201.54 USD
RMSE: 291.64 USD
R2: 0.9081

LSTM
MAE: 74.06 USD
RMSE: 128.66 USD
R2: 0.9821

BILSTM
MAE: 49.62 USD
RMSE: 67.75 USD
R2: 0.9950


- **Modelo MLP:**
Es el modelo que obtiene peores resultados, pero esto puede deberse a que 
- **Modelo LSTM:**
Este modelo obtiene resultados significativamente mejores porque está diseñado para procesar secuencias de datos y recordar información de periodos anteriores. Gracias a ello, puede identificar tendencias y comportamientos del precio del oro a lo largo del tiempo, reduciendo el error de predicción y mejorando la capacidad de ajuste respecto al modelo MLP.
- **Modelo BiLSTM:**
Este es el modelo que ofrece los mejores resultados. Aunque el modelo LSTM ya proporcionaba una precisión muy elevada, se ha utilizado BiLSTM para comprobar si era posible mejorar aún más el rendimiento. Su principal ventaja es que analiza las secuencias en ambos sentidos, lo que le permite capturar relaciones más complejas entre los datos y realizar predicciones más precisas. No obstante, es importante tener en cuenta que los buenos resultados obtenidos también se deben a que muchas de las variables utilizadas están muy relacionadas con el propio precio del oro (apertura, máximo, mínimo, volatilidad o medias móviles), lo que facilita el aprendizaje de patrones por parte del modelo. 

Aun así, el modelo no tiene en cuenta factores externos como conflictos internacionales, inflación, tipos de interés o la evolución del dólar, que también pueden influir en el precio real del oro.

# Guardar datos

In [9]:
results_mlp = pd.DataFrame({
    "date": dates_test,
    "real": y_test_inv.flatten(),
    "pred": pred_mlp_inv.flatten()
})

results_lstm = pd.DataFrame({
    "date": dates_test,
    "real": y_test_inv.flatten(),
    "pred": pred_lstm_inv.flatten()
})

results_bilstm = pd.DataFrame({
    "date": dates_test,
    "real": y_test_inv.flatten(),
    "pred": pred_bilstm_inv.flatten()
})

# Guardar en CSV
results_mlp.to_csv("../datasets/models/mlp_results.csv", index=False)
results_lstm.to_csv("../datasets/models/lstm_results.csv", index=False)
results_bilstm.to_csv("../datasets/models/bilstm_results.csv", index=False)


In [11]:
model_lstm.save("../datasets/models/lstm_model.keras")
model_bilstm.save("../datasets/models/bilstm_model.keras")

model_bilstm.save("../datasets/models/bilstm_model.h5")